# Publish contracts and inspect Kafka delivery semantics

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
from fraudtwin.contracts import contract_registry
from fraudtwin.kafka import publication_records
from fraudtwin.kafka_chaos import KafkaChaosConfig, simulate_delivery

registry = contract_registry()
records = publication_records(data.behavior, run_id, registry=registry)
print({"contracts": len(registry.subjects), "records": len(records)})

**Run the core operation**


In [ ]:
display(
    pl.DataFrame(
        [
            {
                "subject": r.subject,
                "topic": r.topic,
                "record_id": r.record_id,
                "bytes": len(r.value),
            }
            for r in records[:12]
        ]
    )
)

**Measure and interpret the result**


In [ ]:
chaos = simulate_delivery(
    records[:50],
    KafkaChaosConfig(
        seed=7,
        drop_probability=0.05,
        duplicate_probability=0.1,
        retry_probability=0.1,
        max_delay_seconds=30,
        reorder_window=5,
    ),
)
print(chaos.manifest["counts"])

**Exercise a parameter or failure mode**


In [ ]:
assert all(r.record_id for r in chaos.envelopes)
print(
    {
        "input": chaos.input_count,
        "output": chaos.emitted_count,
        "fingerprint": chaos.output_fingerprint,
    }
)

**Write a compact artifact and fingerprint**


In [ ]:
print(
    "Optional: start Kafka + Schema Registry with the commands in the guide; the logical chaos pass above is offline."
)

**Verify invariants and clean up**


In [ ]:
# A compact inspection is more useful than printing an entire run.
print(
    payments.select(
        [
            c
            for c in ("payment_id", "amount", "initiated_at", "payer_account_id")
            if c in payments.columns
        ]
    ).head(8)
)
print({"columns": payments.columns, "nulls": payments.null_count().to_dicts()[0]})

**Optional service integration**


In [ ]:
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))

**Review the expected outcome**


In [ ]:
print("Optional service cell: start Kafka and Schema Registry with the streaming guide.")

## Record the generated shape and tutorial contract.


In [ ]:
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 12,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0